# MAG-S3c: расчёты после релиза v0.3.0

Среда выполнения: **TPU v6e-1** (44 ядра), как в подтверждающем прогоне. Это разведочные расчёты: их план записан в
репозиторий до запуска (коммит 13956c2, раздел 3 файла `mag_s3c/results/exploratory/README.md`), а вердиктов
исследования они не меняют.

1. Ячейка 1 берёт репозиторий на коммите плана, ставит пакеты закреплённых версий и подключает Google Диск.
2. Ячейка 2 сверяет замороженный код с FREEZE.txt, считает 12 контрольных ячеек и сравнивает их с архивом прогона;
   при расхождении она останавливается. Затем она считает 120 ячеек форм P-LH и P-same при углах Kobori et al. и
   8 опорных решений плотной сетки дефазировки, около 1–1,5 часа. Каждые 10 минут готовые ячейки копируются на Диск,
   и после обрыва сессии достаточно снова выполнить ячейки 1 и 2.
3. Ячейка 3 упаковывает результат на Диск частями по 2 МБ и отключает Диск.


In [ ]:
# 1. Репозиторий на коммите плана, пакеты закреплённых версий и Google Диск
import os, subprocess, threading, time
COMMIT = "13956c2740e4b1d227b622a086a7e9a1d8211281"
REPO = "/content/quantum-compass-recording"
WORK = "/content/work/after_release"
DRIVE = "/content/drive/MyDrive/MAG-S3c_after_release"


def sh(cmd, quiet=False):
    if not quiet:
        print("$", cmd, flush=True)
    return subprocess.run(["bash", "-c", cmd]).returncode


if not os.path.exists(REPO):
    assert sh(f"git clone -q https://github.com/nikabri88-cmd/quantum-compass-recording {REPO}") == 0, "git clone"
assert sh(f"cd {REPO} && git fetch -q origin && git checkout -q {COMMIT} && git log -1 --format='%H %s'") == 0
assert sh(f"set -o pipefail; pip install -q -r {REPO}/mag_s3c/code/requirements.txt 2>&1 | tail -3") == 0
sh("python3 -I -c 'import os, sys, numpy, scipy, radicalpy; print(sys.version.split()[0], numpy.__version__, "
   "scipy.__version__, radicalpy.__version__, os.cpu_count(), \"processors\")'")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
print("Диск подключён:", DRIVE)


In [ ]:
# 2. Сверка кода, 12 контрольных ячеек и сравнение с архивом прогона, затем ячейки A и опорные решения B
# (1-1,5 ч на 44 ядрах). Готовые ячейки копируются на Диск: после обрыва сессии выполни ячейки 1 и 2 снова.
if not os.path.exists(WORK) and os.path.exists(f"{DRIVE}/run"):
    sh(f"mkdir -p {os.path.dirname(WORK)} && cp -r {DRIVE}/run {WORK}")
os.makedirs(WORK, exist_ok=True)
stop = threading.Event()


def sync():
    while not stop.wait(600):
        sh(f"mkdir -p {DRIVE}/run && cp -ru {WORK}/. {DRIVE}/run/", quiet=True)


threading.Thread(target=sync, daemon=True).start()
t0 = time.time()
EXP = f"{REPO}/mag_s3c/results/exploratory"
ARCH = f"{REPO}/mag_s3c/results/confirmatory_run/confirmatory_s3c.tar.gz"
rc = sh(f"set -o pipefail; cd {EXP} && python3 -I kobori_and_dephasing.py run {WORK} --archive {ARCH} "
        f"--workers {os.cpu_count()} 2>&1 | tee -a {DRIVE}/after_release.log")
stop.set()
sh(f"mkdir -p {DRIVE}/run && cp -ru {WORK}/. {DRIVE}/run/", quiet=True)
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; журнал: {DRIVE}/after_release.log")
assert rc == 0, "расчёт не завершён: контрольные ячейки не совпали с прогоном или случился сбой, смотри журнал"
print("расчёт завершён, контрольные ячейки совпали с прогоном")


In [ ]:
# 3. Упаковка результата на Диск частями по 2 МБ и отключение Диска
assert os.path.exists(f"{WORK}/done.json"), "расчёт не завершён: сначала ячейка 2"
sh(f"cd {os.path.dirname(WORK)} && tar -czf {DRIVE}/after_release.tar.gz after_release && "
   f"cd {DRIVE} && sha256sum after_release.tar.gz | tee after_release.tar.gz.sha256")
sh(f"cd {DRIVE} && rm -rf parts && mkdir parts && "
   f"split -b 2000000 -d after_release.tar.gz parts/after_release.tar.gz.part && ls -la parts")
drive.flush_and_unmount()
print("готово: части лежат в MAG-S3c_after_release/parts на Google Диске, Диск сброшен и отключён")
